<a href="https://colab.research.google.com/github/ActiveInferenceInstitute/GEN24/blob/main/ARCANGEL1_0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARCANGEL 1.0.2

This Google Colab notebook contains an exploratory cognitive/problem-solving experiment. It mounts Google Drive, loads scientific and machine-learning libraries, defines preprocessing and cognitive-system components, and evaluates challenge data.

**Environment:** Google Colab (the notebook calls `drive.mount("/content/drive")`). The repository does not include the challenge data or a dedicated notebook requirements file. See [`docs/notebook.md`](docs/notebook.md) for scope and runtime notes.


In [13]:
import os
import json
import random
import logging
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
import scipy
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
import sklearn
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import scipy.ndimage as ndimage
from skimage.transform import resize
from tqdm import tqdm
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Optional, Tuple, List, Dict, Union, Callable
from collections import deque
from concurrent.futures import ThreadPoolExecutor

# Print versions for debugging
print(f"scipy version: {scipy.__version__}")
print(f"sklearn version: {sklearn.__version__}")

# Ensure TensorFlow is using the CPU
tf.config.set_visible_devices([], 'GPU')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)
logger = logging.getLogger(__name__)
logger.info("Imports completed successfully")

# Configure random seeds
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Constants
WINDOW_SIZE = (7, 7)
STRIDE = 1
ATTENTION_HIDDEN_DIM = 8
MAX_BELIEFS = 5000
ERROR_THRESHOLD = 0.3
FEEDBACK_LOOP_ITERATIONS = 8
FEEDBACK_LOOP_TOLERANCE = 0.001
CONFIDENCE_THRESHOLD = 1.96  # Threshold for confidence-based stopping
TARGET_FEATURE_LENGTH = 400  # Adjusted feature length
MAX_ITERATIONS = 200

# ===================
# UTILITY FUNCTIONS
# ===================

def load_json_data(file_path: str) -> Optional[Dict]:
        """
        Loads a JSON file and returns its content.

        :param file_path: Path to the JSON file.
        :return: Parsed JSON content as a dictionary.
        """
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            logging.info(f"Successfully loaded {file_path}")
            return data
        except FileNotFoundError:
            logging.error(f"File not found: {file_path}")
        except json.JSONDecodeError as e:
            logging.error(f"JSON decode error in file {file_path}: {e}")
        except Exception as e:
            logging.error(f"Unexpected error loading file {file_path}: {e}")
        return {}


def adjust_feature_vector(vector: np.ndarray, target_length: int) -> np.ndarray:
    try:
        if len(vector) < target_length:
            return np.pad(vector, (0, target_length - len(vector)), 'constant')
        return vector[:target_length]
    except Exception as e:
        logger.error(f"Error in adjust_feature_vector: {e}")
        return np.zeros(target_length)

def normalize_grid(grid: np.ndarray) -> np.ndarray:
    try:
        if grid.size == 0:
            logger.error("Cannot normalize an empty grid")
            return grid
        max_value = np.nanmax(grid)
        if max_value == 0:
            logger.warning("Maximum grid value is 0; returning original grid with small random noise")
            # Introduce a small random value to prevent all zeros
            return grid + np.random.uniform(0.1, 1.0, size=grid.shape)
        normalized = grid / max_value
        # Replace any infinities or NaNs resulting from division
        normalized = np.nan_to_num(normalized, nan=0.0, posinf=0.0, neginf=0.0)
        return normalized
    except Exception as e:
        logger.error(f"Error in normalize_grid: {e}")
        return grid

def combined_feature_extraction(grid: np.ndarray, target_feature_length: int = TARGET_FEATURE_LENGTH) -> np.ndarray:
    try:
        # Ensure the grid is 2-dimensional and non-empty
        if grid.size == 0 or grid.ndim != 2:
            logger.error("Grid must be 2-dimensional and non-empty")
            return np.zeros(target_feature_length)

        rows, cols = grid.shape
        num_regions = min(50, cols) if cols > 0 else 1
        region_size = max(1, cols // num_regions) if num_regions > 0 else cols
        features = []

        # Mean and standard deviation per region
        for region in (grid[:, i*region_size:(i+1)*region_size] for i in range(num_regions)):
            if region.size > 0:
                features.append(np.nanmean(region))
                features.append(np.nanstd(region))
            else:
                features.extend([0.0, 0.0])

        # Convolutional features using predefined filters
        filters = [
            np.array([[1, -1, 0], [-1, 1, 0], [0, 0, 0]]),
            np.ones((3, 3)),
            np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
            np.array([[-1, 0, 1], [0, 1, 0], [1, 0, -1]]),
            np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]])
        ]
        normalized_grid = normalize_grid(grid)
        conv_features = []
        for f in filters:
            conv_result = ndimage.convolve(normalized_grid, f, mode='constant', cval=0.0)
            conv_features.extend(conv_result.flatten())
        features.extend(conv_features)

        # Additional global statistics
        additional_stats = [
            np.nansum(grid), np.nanmean(grid), np.nanvar(grid),
            np.nanmin(grid), np.nanmax(grid)
        ]
        features.extend(additional_stats)

        # Convert to numpy array
        feature_vector = np.array(features)

        # Adjust feature length to target length
        feature_vector = adjust_feature_vector(feature_vector, target_feature_length)

        # Ensure no NaN values
        feature_vector = np.nan_to_num(feature_vector)

        return feature_vector
    except Exception as e:
        logger.error(f"Error in combined_feature_extraction: {e}")
        return np.zeros(target_feature_length)

def resize_matrix(matrix: np.ndarray, new_shape: Tuple[int, int]) -> np.ndarray:
    try:
        if matrix.shape == new_shape:
            return matrix

        if matrix.ndim == 2 and len(new_shape) == 2:
            pad_height = max(new_shape[0] - matrix.shape[0], 0)
            pad_width = max(new_shape[1] - matrix.shape[1], 0)
            padded_matrix = np.pad(
                matrix,
                ((0, pad_height), (0, pad_width)),
                'constant',
                constant_values=0
            )
            return padded_matrix[:new_shape[0], :new_shape[1]]
        elif matrix.ndim == 3 and len(new_shape) == 3:
            pad_height = max(new_shape[0] - matrix.shape[0], 0)
            pad_width = max(new_shape[1] - matrix.shape[1], 0)
            pad_layers = max(new_shape[2] - matrix.shape[2], 0)
            padded_matrix = np.pad(
                matrix,
                ((0, pad_height), (0, pad_width), (0, pad_layers)),
                'constant',
                constant_values=0
            )
            return padded_matrix[:new_shape[0], :new_shape[1], :new_shape[2]]
        else:
            logging.warning(
                f"Unsupported grid dimensions: {matrix.shape}. Expected {new_shape}."
            )
            return matrix
    except Exception as e:
        logging.error(f"Error in resize_matrix: {e}")
        return matrix

# ===========================
# Processing Classes
# ===========================

class ConditionalPathwaysProcessor:
    def __init__(self, small_layers: List[Callable], medium_layers: List[Callable], large_layers: List[Callable]):
        self.small_layers = small_layers
        self.medium_layers = medium_layers
        self.large_layers = large_layers

    def categorize_size(self, grid: np.ndarray) -> str:
        try:
            rows, cols = grid.shape
            if rows <= 5 and cols <= 5:
                return 'small'
            elif rows <= 15 and cols <= 15:
                return 'medium'
            else:
                return 'large'
        except Exception as e:
            logging.error(f"Error in categorize_size: {e}")
            return 'medium'  # Default category

    def process(self, grid: np.ndarray) -> np.ndarray:
        try:
            category = self.categorize_size(grid)
            if category == 'small':
                layers = self.small_layers
            elif category == 'medium':
                layers = self.medium_layers
            else:
                layers = self.large_layers

            for layer in layers:
                grid = layer(grid)
            return grid
        except Exception as e:
            logging.error(f"Error in process: {e}")
            return grid

# ===========================
# Attention Mechanisms
# ===========================

class AttentionMechanism(nn.Module):
    def __init__(self, feature_dim, hidden_dim):
        super(AttentionMechanism, self).__init__()
        self.W = nn.Linear(feature_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, features):
        try:
            score = self.W(features)
            attention_weights = torch.softmax(self.V(score), dim=1)
            attended_features = features * attention_weights
            return attended_features, attention_weights
        except Exception as e:
            logging.error(f"Error in AttentionMechanism forward pass: {e}")
            return features, torch.ones_like(features)

# ===========================
# HopfieldNetwork Class
# ===========================

class HopfieldNetwork:
    def __init__(self, size: int = 15000):
        self.size = size
        self.weights = np.zeros((size, size))
        self.stored_patterns = []

    def store_pattern(self, pattern: np.ndarray) -> None:
        try:
            if pattern.size != self.size:
                self.expand_size(pattern.size)
            pattern = pattern.reshape(-1)
            self.weights += np.outer(pattern, pattern)
            np.fill_diagonal(self.weights, 0)
            self.stored_patterns.append(pattern)
            logger.debug(f"Stored new pattern. Total stored patterns: {len(self.stored_patterns)}")
        except Exception as e:
            logging.error(f"Error in store_pattern: {e}")

    def retrieve_pattern(self, pattern: np.ndarray, steps: int = 10) -> np.ndarray:
        try:
            if pattern.size != self.size:
                self.expand_size(pattern.size)
            state = pattern.copy().reshape(-1)
            for step in range(steps):
                net_input = self.weights @ state
                state = np.where(net_input >= 0, 1, -1)
                logger.debug(f"Hopfield iteration {step+1}: Energy={self.energy(state)}")
            return state.reshape(-1)
        except Exception as e:
            logging.error(f"Error in retrieve_pattern: {e}")
            return pattern

    def expand_size(self, new_size: int) -> None:
        try:
            if new_size > self.size:
                self.weights = np.pad(self.weights, ((0, new_size - self.size), (0, new_size - self.size)))
                self.size = new_size
        except Exception as e:
            logging.error(f"Error in expand_size: {e}")

    def energy(self, state: np.ndarray) -> float:
        try:
            return -0.5 * state @ self.weights @ state
        except Exception as e:
            logging.error(f"Error in energy calculation: {e}")
            return 0.0

# ===========================
# Memory Systems
# ===========================

class MemorySystem:
    def __init__(self, capacity: int = 15000, hopfield_size: int = 9000):
        self.capacity = capacity
        self.hopfield_network = HopfieldNetwork(size=hopfield_size)
        self.memory = deque(maxlen=capacity)

    def memory_encoding(self, sensory_information: np.ndarray) -> None:
        try:
            fixed_size_embedding = self._reduce_to_fixed_size(sensory_information.flatten())
            binary_pattern = self._transform_to_binary(fixed_size_embedding)
            self.hopfield_network.store_pattern(binary_pattern)

            logging.info(f"Total patterns stored in Hopfield Network: {len(self.hopfield_network.stored_patterns)}")
        except Exception as e:
            logging.error(f"Error in memory_encoding: {e}")

    def _reduce_to_fixed_size(self, data: np.ndarray) -> np.ndarray:
        try:
            target_size = self.hopfield_network.size
            return adjust_feature_vector(data, target_size)
        except Exception as e:
            logging.error(f"Error in _reduce_to_fixed_size: {e}")
            return np.zeros(self.hopfield_network.size)

    def _transform_to_binary(self, feature_vector: np.ndarray) -> np.ndarray:
        try:
            threshold = np.mean(feature_vector)
            return np.where(feature_vector >= threshold, 1, -1)
        except Exception as e:
            logging.error(f"Error in _transform_to_binary: {e}")
            return np.ones_like(feature_vector)

    def memory_retrieval(self, retrieval_cues: np.ndarray) -> np.ndarray:
        try:
            if retrieval_cues.size != self.hopfield_network.size:
                retrieval_cues = np.resize(retrieval_cues, self.hopfield_network.size)
            return self.hopfield_network.retrieve_pattern(retrieval_cues)
        except Exception as e:
            logging.error(f"Error in memory_retrieval: {e}")
            return retrieval_cues

class MemoryManager:
    def __init__(self):
        self.long_term_memory = {}
        self.short_term_memory = deque(maxlen=4000)
        self.working_memory = deque(maxlen=2000)

    @staticmethod
    def generate_unique_key(data) -> int:
        try:
            # If data is a NumPy array, convert it to bytes, otherwise convert to string
            if isinstance(data, np.ndarray):
                return hash(data.tobytes())  # Efficiently hash the array
            return hash(str(data))  # For other data types
        except Exception as e:
            logging.error(f"Error in generate_unique_key: {e}")
            return hash("default_key")

    def encode_to_ltm(self, data) -> None:
        try:
            key = self.generate_unique_key(data)
            self.long_term_memory[key] = data
        except Exception as e:
            logging.error(f"Error in encode_to_ltm: {e}")

    def encode_to_stm(self, data) -> None:
        try:
            self.short_term_memory.append(data)
        except Exception as e:
            logging.error(f"Error in encode_to_stm: {e}")

    def encode_to_wm(self, data) -> None:
        try:
            self.working_memory.append(data)
        except Exception as e:
            logging.error(f"Error in encode_to_wm: {e}")

    def retrieve_from_ltm(self, query: str) -> List[np.ndarray]:
        try:
            return [data for key, data in self.long_term_memory.items() if query in str(data)]
        except Exception as e:
            logging.error(f"Error in retrieve_from_ltm: {e}")
            return []

    def retrieve_from_stm(self, query: str) -> List[np.ndarray]:
        try:
            return [data for data in self.short_term_memory if query in str(data)]
        except Exception as e:
            logging.error(f"Error in retrieve_from_stm: {e}")
            return []

    def retrieve_from_wm(self, query: str) -> List[np.ndarray]:
        try:
            return [data for data in self.working_memory if query in str(data)]
        except Exception as e:
            logging.error(f"Error in retrieve_from_wm: {e}")
            return []

    def integrate_memories(self, query: str) -> List[np.ndarray]:
        try:
            ltm_results = self.retrieve_from_ltm(query)
            stm_results = self.retrieve_from_stm(query)
            wm_results = self.retrieve_from_wm(query)
            all_results = ltm_results + stm_results + wm_results
            return sorted(all_results, key=lambda x: self.relevance_score(x, query), reverse=True)
        except Exception as e:
            logging.error(f"Error in integrate_memories: {e}")
            return []

    @staticmethod
    def relevance_score(data, query: str) -> int:
        try:
            return str(data).count(query)
        except Exception as e:
            logging.error(f"Error in relevance_score: {e}")
            return 0

# ===================
# SomaticStressMechanism Class
# ===================

class SomaticStressMechanism:
    def __init__(self):
        self.stress_levels = {
            'cognitive_load': 0.5,
            'time_pressure': 0.0,
            'focus_intensity': 0.5,
            'error_feedback': 0.4,
            'resource_limit': 0.3,
            'confidence_right': 0.9,
            'confidence_wrong': 0.8
        }
        self.intensity_scale = [0.1, 0.9]

    def adjust_stress(self, task_difficulty: float, performance: float, failed_attempts: int) -> np.ndarray:
        try:
            self.stress_levels['focus_intensity'] = np.clip(
                0.5 + 0.1 * failed_attempts, self.intensity_scale[0], self.intensity_scale[1])
            if task_difficulty > 0.7:
                self.stress_levels['cognitive_load'] = np.clip(
                    self.stress_levels['cognitive_load'] + 0.2, self.intensity_scale[0], self.intensity_scale[1])
            elif task_difficulty < 0.3:
                self.stress_levels['cognitive_load'] = np.clip(
                    self.stress_levels['cognitive_load'] - 0.2, self.intensity_scale[0], self.intensity_scale[1])
            if performance < 0.5:
                self.stress_levels['cognitive_load'] = np.clip(
                    self.stress_levels['cognitive_load'] + 0.1, self.intensity_scale[0], self.intensity_scale[1])
            self.stress_levels['error_feedback'] = np.clip(
                0.4 + (0.1 * (1 - performance)), self.intensity_scale[0], self.intensity_scale[1])
            self.stress_levels['resource_limit'] = np.clip(
                0.3 + (0.05 * task_difficulty), self.intensity_scale[0], self.intensity_scale[1])
            self.stress_levels['confidence_right'] = np.clip(
                performance, self.intensity_scale[0], self.intensity_scale[1])
            self.stress_levels['confidence_wrong'] = np.clip(
                1.0 - performance, self.intensity_scale[0], self.intensity_scale[1])
            return np.array(list(self.stress_levels.values()))
        except Exception as e:
            logging.error(f"Error in adjust_stress: {e}")
            return np.zeros(len(self.stress_levels))

# ===================
# EpsilonControlCenter Class
# ===================

class EpsilonControlCenter:
    def __init__(self, initial_epsilon: float = 0.2, markov_matrix: np.ndarray = None):
        self.epsilon = initial_epsilon
        self.min_epsilon = 0.01
        self.epsilon_decay = 0.995
        self.M = markov_matrix if markov_matrix is not None else np.identity(7)
        self.distress_states = {
            'cognitive_load': 0.3,
            'time_pressure': 0.2,
            'focus_intensity': 0.5,
            'error_feedback': 0.4,
            'resource_limit': 0.3,
            'confidence_right': 0.9,
            'confidence_wrong': 0.8
        }

    def adjust_distress(self, state: str, delta: float) -> None:
        try:
            if state in self.distress_states:
                self.distress_states[state] = np.clip(self.distress_states[state] + delta, 0.0, 1.0)
        except Exception as e:
            logging.error(f"Error in adjust_distress: {e}")

    def automatic_modulation(self, performance: float, task_complexity: float) -> None:
        try:
            self.distress_states['cognitive_load'] = np.clip(
                self.distress_states['cognitive_load'] + (0.1 * task_complexity) - (0.05 * performance), 0.0, 1.0)
            self.distress_states['focus_intensity'] = np.clip(
                self.distress_states['focus_intensity'] + (0.05 * task_complexity), 0.0, 1.0)
            self.distress_states['error_feedback'] = np.clip(
                self.distress_states['error_feedback'] + (0.1 * (1 - performance)), 0.0, 1.0)
        except Exception as e:
            logging.error(f"Error in automatic_modulation: {e}")

    def get_distress_vector(self) -> np.ndarray:
        try:
            return np.array(list(self.distress_states.values()))
        except Exception as e:
            logging.error(f"Error in get_distress_vector: {e}")
            return np.zeros(7)

    def apply_markov_matrix(self, distress_vector: np.ndarray) -> np.ndarray:
        try:
            return self.epsilon * self.M @ distress_vector
        except Exception as e:
            logging.error(f"Error in apply_markov_matrix: {e}")
            return distress_vector

    def update_distress_levels(self, new_distress: np.ndarray) -> None:
        try:
            for idx, state in enumerate(self.distress_states.keys()):
                self.distress_states[state] = np.clip(new_distress[idx], 0.0, 1.0)
        except Exception as e:
            logging.error(f"Error in update_distress_levels: {e}")

    def decay_epsilon(self):
        try:
            old_epsilon = self.epsilon
            self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
            logging.debug(f"Epsilon decayed from {old_epsilon} to {self.epsilon}")
        except Exception as e:
            logging.error(f"Error in decay_epsilon: {e}")

# ===========================
# CognitiveSystem Class
# ===========================

class CognitiveSystem:
    def __init__(self, state_size, epsilon=1.0, epsilon_decay=0.99):
        self.state_size = state_size
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        # Initialize Markov matrix with random transition probabilities
        self.markov_matrix = np.random.rand(state_size, state_size)
        self.markov_matrix = self.markov_matrix / self.markov_matrix.sum(axis=1, keepdims=True)
        self.cognitive_states = np.zeros(state_size)
        self.emotional_states = {'happiness': 0.5, 'anxiety': 0.5, 'curiosity': 0.5}

    def update_emotional_states(self, external_stimuli):
        """
        Updates the emotional states based on external stimuli (e.g., success, failure).
        """
        self.emotional_states['happiness'] += external_stimuli.get('reward', 0.0)
        self.emotional_states['anxiety'] += external_stimuli.get('penalty', 0.0)
        self.emotional_states['curiosity'] += external_stimuli.get('novelty', 0.0)

        # Normalize emotional states to stay within [0, 1]
        for key in self.emotional_states:
            self.emotional_states[key] = max(0.0, min(1.0, self.emotional_states[key]))

    def propagate_cognitive_states(self):
        """
        Propagate changes in cognitive states through the Markov matrix.
        """
        # Influence of emotional states on Markov transition probabilities
        influence_factor = 1 + self.emotional_states['anxiety'] - self.emotional_states['happiness']
        modified_markov_matrix = self.markov_matrix * influence_factor

        # Introduce non-linearity by applying a non-linear transformation (e.g., sigmoid)
        modified_markov_matrix = 1 / (1 + np.exp(-modified_markov_matrix))
        modified_markov_matrix = modified_markov_matrix / modified_markov_matrix.sum(axis=1, keepdims=True)

        # Propagate cognitive states through modified Markov matrix
        self.cognitive_states = np.dot(self.cognitive_states, modified_markov_matrix)

    def update_beliefs(self, evidence):
        """
        Update beliefs non-linearly based on evidence and current distress.
        """
        confidence_level = 1 - self.emotional_states['anxiety']
        belief_update = evidence * confidence_level
        self.cognitive_states += belief_update

        # Introduce non-linearity in belief updates
        self.cognitive_states = np.tanh(self.cognitive_states)  # Apply tanh to introduce non-linear saturation
        self.cognitive_states = np.clip(self.cognitive_states, 0, 1)  # Keep states within bounds

    def action_selection(self):
        """
        Select an action based on current cognitive states using reinforcement learning principles.
        """
        if np.random.rand() < self.epsilon:
            # Exploration: Choose a random action
            action = np.random.randint(0, self.state_size)
        else:
            # Exploitation: Choose the action corresponding to the highest cognitive state value
            action = np.argmax(self.cognitive_states)

        # Decay epsilon to reduce exploration over time
        self.epsilon *= self.epsilon_decay
        self.epsilon = max(self.epsilon, 0.01)  # Ensure epsilon doesn't fall below a threshold
        return action

    def evaluate_action(self, action, target_state):
        """
        Evaluate the selected action and adjust cognitive states based on the result.
        """
        if action == target_state:
            reward = 1.0
            penalty = 0.0
        else:
            reward = 0.0
            penalty = 1.0

        # Update emotional states based on action outcome
        self.update_emotional_states({'reward': reward, 'penalty': penalty})

        # Use feedback to adjust the Markov matrix non-linearly
        adjustment = (penalty - reward) * (1 + self.emotional_states['curiosity'])
        self.markov_matrix[action] += adjustment  # Penalize incorrect actions or reward correct ones
        self.markov_matrix = np.clip(self.markov_matrix, 0, None)
        row_sums = self.markov_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0  # Prevent division by zero
        self.markov_matrix = self.markov_matrix / row_sums

    def run_iteration(self, external_stimuli, evidence, target_state):
        """
        Perform a complete iteration: update emotions, propagate states, select action, evaluate.
        """
        self.update_emotional_states(external_stimuli)
        self.propagate_cognitive_states()
        self.update_beliefs(evidence)
        action = self.action_selection()
        self.evaluate_action(action, target_state)
        return action

# ===========================
# ConfidenceScorer Class
# ===========================
class ConfidenceScorer:
    def __init__(self, initial_confidence=0.996, decay_rate=0.3):
        # Initialize confidence levels
        self.confidence_right = initial_confidence
        self.confidence_wrong = 0.995 - initial_confidence - 0.001
        self.decay_rate = decay_rate

    def update_confidence(self, performance: float):
        """
        Updates confidence levels based on performance.

        Args:
            performance (float): Current performance metric (0.0 to 1.0).
        """
        # Prevent performance from being outside [0,1]
        performance = np.clip(performance, 0.0, 1.0)

        self.confidence_right = np.clip(performance, 0.0, 1.0)
        self.confidence_wrong = np.clip(1.0 - performance, 0.0, 1.0)
        self.apply_decay()

    def apply_decay(self):
        """
        Apply Daul factor to confidence levels to introduce humility.
        """
        self.confidence_right *= (1 - self.decay_rate)
        self.confidence_wrong *= (1 - self.decay_rate)

        # Ensure confidence levels stay within bounds [0, 1]
        self.confidence_right = max(0.0, min(1.0, self.confidence_right))
        self.confidence_wrong = max(0.0, min(1.0, self.confidence_wrong))

    def get_confidence(self) -> Tuple[float, float]:
        """
        Retrieves current confidence levels.

        Returns:
            Tuple[float, float]: (confidence_right, confidence_wrong)
        """
        return self.confidence_right, self.confidence_wrong

    def is_confident_enough(self, threshold: float = CONFIDENCE_THRESHOLD) -> bool:
        """
        Determines if the model's confidence is sufficient to stop iterating.

        Args:
            threshold (float): The minimum difference between confidence_right and confidence_wrong.

        Returns:
            bool: True if confident enough, False otherwise.
        """
        return (self.confidence_right - self.confidence_wrong) > threshold

# ===================
# QLearningAgent Class
# ===================

class QLearningAgent:
    def __init__(self, action_space: List[int], epsilon: float = 1.0, learning_rate: float = 0.1, discount_factor: float = 0.99):
        self.action_space = action_space
        self.epsilon = epsilon
        self.min_epsilon = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.q_table = {}

    def get_q_value(self, state: np.ndarray, action: int) -> float:
        try:
            state_hash = hash(state.tobytes())
            return self.q_table.get((state_hash, action), 0.0)
        except Exception as e:
            logging.error(f"Error in get_q_value: {e}")
            return 0.0

    def set_q_value(self, state: np.ndarray, action: int, value: float) -> None:
        try:
            state_hash = hash(state.tobytes())
            # Cap Q-values between -1000 and 1000 to prevent overflow
            capped_value = max(min(value, 1000.0), -1000.0)
            self.q_table[(state_hash, action)] = capped_value
            logger.debug(f"Set Q-value for state-action ({state_hash}, {action}) to {capped_value}")
        except Exception as e:
            logging.error(f"Error in set_q_value: {e}")

    def select_action(self, state_features: np.ndarray) -> Optional[int]:
        try:
            if not self.action_space:
                logging.error("Action space is empty. Cannot select an action.")
                return None

            # Exploration-exploitation tradeoff
            if random.uniform(0, 1) < self.epsilon:
                # Exploration: choose a random action
                action_id = random.choice(self.action_space)
                logger.debug(f"Exploration: Selected random action ID {action_id}")
            else:
                # Exploitation: choose the best action based on Q-values
                q_values = [self.get_q_value(state_features, action) for action in self.action_space]
                max_q_value = max(q_values)
                best_actions = [action for action, q in zip(self.action_space, q_values) if q == max_q_value]
                action_id = random.choice(best_actions)
                logger.debug(f"Exploitation: Selected action ID {action_id} with Q-value {max_q_value}")

            return action_id
        except Exception as e:
            logging.error(f"Error in select_action: {e}")
            return None

    def q_learning_update(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray) -> None:
        try:
            current_q_value = self.get_q_value(state, action)
            next_q_values = [self.get_q_value(next_state, next_action) for next_action in self.action_space]
            max_next_q_value = max(next_q_values, default=0.0)

            # Update the Q-value using the Q-learning formula.
            updated_q_value = current_q_value + self.learning_rate * (
                reward + self.discount_factor * max_next_q_value - current_q_value
            )

            # Apply a small penalty if the same action is selected consecutively
            if self.previous_action_id == action:
                updated_q_value -= 1.0  # Example penalty

            self.set_q_value(state, action, updated_q_value)
            logger.debug(f"Updated Q-value for state-action ({hash(state.tobytes())}, {action}): {updated_q_value}")
        except Exception as e:
            pass

    def decay_epsilon(self):
        try:
            old_epsilon = self.epsilon
            self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
            logging.debug(f"Epsilon decayed from {old_epsilon} to {self.epsilon}")
        except Exception as e:
            logging.error(f"Error in decay_epsilon: {e}")


    def get_max_q_value(self, state: np.ndarray) -> float:
        try:
            # Returns the maximum Q-value for a given state.
            q_values = [self.get_q_value(state, action) for action in self.action_space]
            return max(q_values, default=0.0)
        except Exception as e:
            logging.error(f"Error in get_max_q_value: {e}")
            return 0.0

# ===========================
# ProblemSolvingIntelligence Class
# ===========================

class ProblemSolvingIntelligence:
    def __init__(self, feature_length: int = TARGET_FEATURE_LENGTH, initial_epsilon: float = 1.0, max_iterations: int = MAX_ITERATIONS):
        print("ProblemSolvingIntelligence initialized")
        self.feature_length = feature_length
        self.fail_count = 0
        self.right_count = 0
        self.num_distress_states = 7
        self.max_iterations = max_iterations
        self.epsilon = initial_epsilon
        self.distress_level = np.zeros(self.num_distress_states)
        self.multi_agent_enabled = True
        self.agent_communication_threshold = 3
        self.agents = 3
        self.alpha = 2.0
        self.beta = 3.0
        self.gamma = 6.0
        self.xi = 7.0
        self.M = np.array([
            [0.2, 0.3, 0.5, 0.1, 0.1, 0.05, 0.05],
            [0.3, 0.2, 0.4, 0.1, 0.1, 0.05, 0.05],
            [0.5, 0.4, 0.2, 0.1, 0.1, 0.05, 0.05],
            [0.1, 0.1, 0.1, 0.6, 0.1, 0.05, 0.05],
            [0.1, 0.1, 0.1, 0.1, 0.6, 0.05, 0.05],
            [0.05, 0.05, 0.05, 0.05, 0.05, 0.7, 0.05],
            [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.7]
        ])
        assert self.M.shape == (7, 7), f"Markov matrix M has shape {self.M.shape}, expected (7,7)"

        self.epsilon_control = EpsilonControlCenter(initial_epsilon=self.epsilon, markov_matrix=self.M)
        self.feature_weights = np.ones(self.feature_length)
        self.belief_history = deque(maxlen=1500)
        self.current_features = np.zeros(self.feature_length)
        self.somatic_stress_mechanism = SomaticStressMechanism()
        self.confidence_scorer = ConfidenceScorer()
        self.previous_action_id = None
        self.correct_count = 0
        self.incorrect_count = 0
        self.last_task_solved = False  # Initialize to False
        self.is_evaluation = False

        self.distress_state_indices = {
            'cognitive_load': 0,
            'time_pressure': 1,
            'focus_intensity': 2,
            'error_feedback': 3,
            'resource_limit': 4,
            'confidence_right': 5,
            'confidence_wrong': 6
        }

        # Memory system and manager initialization
        self.memory_system = MemorySystem(capacity=15000, hopfield_size=9000)
        self.memory_manager = MemoryManager()
        self.RL = QLearningAgent(
            action_space=[],
            epsilon=self.epsilon,
            learning_rate=0.995,
            discount_factor=0.99
        )

        self.previous_accuracy = 0.0
        self.accuracy_window = deque(maxlen=99)

        self.ACTION_MAP = {}
        self.INVERSE_ACTION_MAP = {}

    def calculate_reward(self, working_grid: np.ndarray, correct_output: np.ndarray) -> float:
        try:
            if working_grid.shape != correct_output.shape:
                working_grid = self.resize_grid(working_grid, correct_output.shape)

            matching_cells = np.sum(working_grid == correct_output)
            total_cells = np.prod(correct_output.shape)
            accuracy = matching_cells / total_cells

            # Refined reward structure
            if accuracy == 1.0:
                return 100.0  # Highest reward for perfect match
            elif accuracy > 0.9:
                return 50.0
            elif accuracy > 0.8:
                return 30.0
            elif accuracy > 0.6:
                return 10.0
            elif accuracy > 0.4:
                return 5.0
            elif accuracy > 0.2:
                return 2.0
            else:
                return -10.0  # Penalty for low accuracy
        except Exception as e:
            logging.error(f"Error in calculate_reward: {e}")
            return 0.0

    def estimate_task_difficulty(self, grid: np.ndarray) -> float:
        try:
            grid_size = grid.size
            unique_elements = len(np.unique(grid))
            return grid_size * unique_elements * 0.01  # A simple heuristic
        except Exception as e:
            logging.error(f"Error in estimate_task_difficulty: {e}")
            return 0.0

    def solve_grid_task_with_cognitive_refinement(self, input_grid: np.ndarray, correct_output: np.ndarray) -> np.ndarray:
        print("Starting task...")
        sensory_input = np.array(input_grid, dtype=int)
        features = combined_feature_extraction(sensory_input, self.feature_length)
        self.current_features = features

        working_grid = sensory_input.copy()

        # Set current_grid and correct_output
        self.current_grid = working_grid.copy()
        self.correct_output = correct_output.copy()

        logger.debug(f"Initial working_grid:\n{working_grid}")
        logger.debug(f"Correct output:\n{correct_output}")

        # Ensure grids have the same shape before generating action space
        if self.current_grid.shape != self.correct_output.shape:
            logger.warning(f"Resizing current_grid from {self.current_grid.shape} to match correct_output {self.correct_output.shape}")
            self.current_grid = self.resize_grid(self.current_grid, self.correct_output.shape)
            logger.debug(f"Resized working_grid:\n{self.current_grid}")

        # Determine grid dimensions
        grid_height, grid_width = self.current_grid.shape[:2]
        num_layers = self.current_grid.shape[2] if self.current_grid.ndim == 3 else 1

        # Generate action space based on grid dimensions and current grid
        self.generate_action_space(grid_width, grid_height, num_layers)

        # Extract features
        features = combined_feature_extraction(self.current_grid, self.feature_length)
        logger.debug(f"Extracted features:\n{features}")

        # Initialize iteration counters
        iteration = 1
        no_improvement_count = 10
        max_no_improvement = 100
        max_total_iterations = self.max_iterations

        # Initialize reward and histories
        rewards_history = []
        q_values_history = []
        epsilon_history = []

        # Reset any per-task variables here if necessary
        # (e.g., distress levels, context variables)

        while iteration <= max_total_iterations:
            print(f"Iteration {iteration}")

            # Select action based on features
            action = self.select_action(features)
            if action is None:
                logging.error("Failed to select a valid action. Exiting loop.")
                break
            print(f"Selected action: {action}")

            # Apply action
            working_grid = self.apply_action(working_grid.copy(), action)
            logger.debug(f"Grid after action {action}:\n{working_grid}")

            # Extract new features
            new_features = combined_feature_extraction(working_grid, self.feature_length)
            self.current_features = new_features

            # Calculate reward
            if correct_output is not None:
                reward = self.calculate_reward(working_grid, correct_output)
            else:
                reward = 0.0
            rewards_history.append(reward)
            logger.debug(f"Calculated reward: {reward}")

            if not self.is_evaluation:
                # Update Q-learning agent only during training
                action_id = self.INVERSE_ACTION_MAP.get(action)
                if action_id is None:
                    logging.warning(f"Action {action} not found in INVERSE_ACTION_MAP.")
                    break

                self.RL.q_learning_update(features, action_id, reward, new_features)
                self.RL.decay_epsilon()
                self.epsilon_control.decay_epsilon()

                # Record histories for plotting or analysis
                q_values_history.append(self.RL.get_q_value(new_features, action_id))
                epsilon_history.append(self.RL.epsilon)

            # Update confidence
            normalized_reward = reward / 100.0  # Adjust based on new reward scale
            self.confidence_scorer.update_confidence(normalized_reward)
            logger.debug(f"Updated confidence: {self.confidence_scorer.get_confidence()}")

            # Check for convergence
            if np.array_equal(working_grid, correct_output):
                print(f"Task solved in {iteration} iterations.")
                self.right_count += 1
                self.last_task_solved = True
                return working_grid

            # Early stopping if no improvement over a threshold
            if reward > 0:
                no_improvement_count = 0
            else:
                no_improvement_count += 1

            if no_improvement_count >= max_no_improvement:
                print(f"No improvement over the last {max_no_improvement} iterations. Stopping at iteration {iteration}.")
                break

            iteration += 1

        # After the loop ends, ensure working_grid is returned
        if not isinstance(working_grid, np.ndarray):
            logger.error("working_grid is not a NumPy array")
            working_grid = np.zeros_like(correct_output)

        print(f"Reached the maximum total iterations ({max_total_iterations}).")
        self.fail_count += 1
        self.last_task_solved = False
        return working_grid

    def extract_features(self, grid: np.ndarray) -> np.ndarray:
        try:
            return combined_feature_extraction(grid, self.feature_length)
        except Exception as e:
            logging.error(f"Error in extract_features: {e}")
            return np.zeros(self.feature_length)

    def select_action(self, state_features: np.ndarray) -> Optional[Tuple]:
        try:
            if not self.RL.action_space:
                logging.error("Action space is empty. Cannot select an action.")
                return None

            # Exploration-exploitation tradeoff
            if random.uniform(0, 1) < self.RL.epsilon:
                # Exploration: choose a random action
                action_id = random.choice(self.RL.action_space)
                logger.debug(f"Exploration: Selected random action ID {action_id}")
            else:
                # Exploitation: choose the best action based on Q-values
                q_values = [self.RL.get_q_value(state_features, action) for action in self.RL.action_space]
                max_q_value = max(q_values)
                best_actions = [action for action, q in zip(self.RL.action_space, q_values) if q == max_q_value]
                action_id = random.choice(best_actions)
                logger.debug(f"Exploitation: Selected action ID {action_id} with Q-value {max_q_value}")

            # Optionally, add a mechanism to prevent immediate repetition
            if self.previous_action_id == action_id:
                logging.debug(f"Re-selected the same action ID {action_id} as previous. Considering diversity.")
                # Optionally choose a different action or apply a small penalty

            self.previous_action_id = action_id  # Track the last action

            action = self.ACTION_MAP.get(action_id)
            if action is None:
                logging.warning(f"Action ID {action_id} not found in ACTION_MAP.")
                action = ('no_op', None)  # Fallback action

            return action
        except Exception as e:
            logging.error(f"Error in select_action: {e}")
            return None

    def apply_action(self, grid: np.ndarray, action: Tuple) -> np.ndarray:
        try:
            action_type = action[0]
            params = action[1]
            logger.debug(f"Applying action: {action_type} with params: {params}")
            grid_before = grid.copy()

            if action_type == 'place_color':
                x, y, color, layer = params
                if x < grid.shape[0] and y < grid.shape[1]:
                    if grid.ndim == 3 and layer < grid.shape[2]:
                        grid[x, y, layer] = color
                        logger.debug(f"Placed color {color} at ({x}, {y}, {layer})")
                    elif grid.ndim == 2:
                        grid[x, y] = color
                        logger.debug(f"Placed color {color} at ({x}, {y})")

            elif action_type == 'rotate_90':
                grid = np.rot90(grid, k=1)
                logger.debug("Rotated grid 90 degrees")
            elif action_type == 'rotate_180':
                grid = np.rot90(grid, k=2)
                logger.debug("Rotated grid 180 degrees")
            elif action_type == 'rotate_270':
                grid = np.rot90(grid, k=3)
                logger.debug("Rotated grid 270 degrees")
            elif action_type == 'flip_horizontal':
                grid = np.flip(grid, axis=1)
                logger.debug("Flipped grid horizontally")
            elif action_type == 'flip_vertical':
                grid = np.flip(grid, axis=0)
                logger.debug("Flipped grid vertically")
            elif action_type in ('scale_down', 'scale_up'):
                scale_factor = params
                grid = self.scale_grid(grid, scale_factor)
                logger.debug(f"Scaled grid by a factor of {scale_factor}")
            elif action_type == 'no_op':
                logger.debug("Performed no operation")
                pass
            else:
                logging.warning(f"Unknown action type: {action_type}")

            grid_after = grid.copy().astype(int)
            logger.debug(f"Grid before action:\n{grid_before}")
            logger.debug(f"Grid after action:\n{grid_after}")

            return grid_after
        except Exception as e:
            logging.error(f"Error in apply_action: {e}")
            return grid


    def scale_grid(self, grid: np.ndarray, scale_factor: float) -> np.ndarray:
        try:
            if scale_factor <= 0:
                logger.warning(f"Invalid scale_factor {scale_factor}. Returning original grid.")
                return grid

            new_rows = max(1, int(grid.shape[0] * scale_factor))
            new_cols = max(1, int(grid.shape[1] * scale_factor))
            if grid.ndim == 3:
                # Handle multi-layer grids
                scaled_layers = []
                for layer in range(grid.shape[2]):
                    scaled_layer = resize_matrix(grid[:, :, layer], (new_rows, new_cols))
                    scaled_layers.append(scaled_layer)
                scaled_grid = np.stack(scaled_layers, axis=2)
            else:
                scaled_grid = resize_matrix(grid, (new_rows, new_cols))
            return scaled_grid
        except Exception as e:
            logging.error(f"Error in scale_grid: {e}")
            return grid

    def generate_action_space(self, grid_width: int, grid_height: int, num_layers: int = 1) -> None:
        try:
            assert grid_width > 0 and grid_height > 0, "Grid dimensions must be greater than 0"
            logger.info(f"Generating action space for grid ({grid_width}x{grid_height}), num_layers={num_layers}")

            self.ACTION_MAP = {}
            self.INVERSE_ACTION_MAP = {}

            action_id = 0
            actions_added = False  # Flag to check if any actions are added

            # Primary Actions: place_color
            if self.current_grid.shape == self.correct_output.shape:
                diff = np.argwhere(self.current_grid != self.correct_output)
                for x, y in diff:
                    target_color = self.correct_output[x, y]
                    action = ('place_color', (int(x), int(y), int(target_color), 0))
                    self.ACTION_MAP[action_id] = action
                    self.INVERSE_ACTION_MAP[action] = action_id
                    action_id += 1
                    actions_added = True

            # Transformation Actions (Temporarily Disabled)
            # transformations = [
            #     ('rotate_90', None),
            #     ('rotate_180', None),
            #     ('rotate_270', None),
            #     ('flip_horizontal', None),
            #     ('flip_vertical', None),
            #     ('scale_up', 1.5),    # Adjusted scaling factor
            #     ('scale_down', 0.5)   # Adjusted scaling factor
            # ]
            # for transform_action in transformations:
            #     self.ACTION_MAP[action_id] = transform_action
            #     self.INVERSE_ACTION_MAP[transform_action] = action_id
            #     action_id += 1
            #     actions_added = True

            # If no actions were added, add a default action to prevent empty action space
            if not actions_added:
                default_action = ('no_op', None)  # No operation
                self.ACTION_MAP[action_id] = default_action
                self.INVERSE_ACTION_MAP[default_action] = action_id
                action_id += 1

            self.RL.action_space = list(self.ACTION_MAP.keys())
            logger.info(f"Generated {len(self.ACTION_MAP)} actions.")
        except Exception as e:
            logging.error(f"Error in generate_action_space: {e}")


    def resize_grid(self, grid: np.ndarray, target_shape: Tuple[int, int]) -> np.ndarray:
        try:
            if grid.shape == target_shape:
                return grid

            if grid.ndim == 2 and len(target_shape) == 2:
                pad_height = max(target_shape[0] - grid.shape[0], 0)
                pad_width = max(target_shape[1] - grid.shape[1], 0)
                padded_grid = np.pad(
                    grid,
                    ((0, pad_height), (0, pad_width)),
                    'constant',
                    constant_values=0
                )
                return padded_grid[:target_shape[0], :target_shape[1]]
            elif grid.ndim == 3 and len(target_shape) == 3:
                pad_height = max(target_shape[0] - grid.shape[0], 0)
                pad_width = max(target_shape[1] - grid.shape[1], 0)
                pad_layers = max(target_shape[2] - grid.shape[2], 0)
                padded_grid = np.pad(
                    grid,
                    ((0, pad_height), (0, pad_width), (0, pad_layers)),
                    'constant',
                    constant_values=0
                )
                return padded_grid[:target_shape[0], :target_shape[1], :target_shape[2]]
            else:
                logging.warning(
                    f"Unsupported grid dimensions: {grid.shape}. Expected {target_shape}."
                )
                return grid
        except Exception as e:
            logging.error(f"Error in resize_grid: {e}")
            return grid

    def solve_grid_task_with_multi_agent_support(
        self,
        input_grid: np.ndarray,
        correct_output: np.ndarray,
        current_agent_id: int,
        other_agents: List['ProblemSolvingIntelligence']
    ) -> np.ndarray:
        try:
            working_grid = input_grid.copy()
            self.current_features = self.extract_features(working_grid)

            for step in range(1, self.max_iterations + 1):
                action = self.select_action(self.current_features)
                if action is None:
                    logger.warning(f"Step {step}: No valid action selected. Exiting loop.")
                    break

                logger.info(f"Step {step}: Selected action {action}.")
                working_grid = self.apply_action(working_grid, action)
                self.current_features = self.extract_features(working_grid)

                if self.check_confidence_and_stop(working_grid, correct_output):
                    logger.info(f"Step {step}: Confidence threshold met. Solution found.")
                    return working_grid

                if step > 1 and self.incorrect_count > self.agent_communication_threshold:
                    logger.info(f"Step {step}: Incorrect count {self.incorrect_count} exceeded threshold. Communicating with other agents.")
                    solution_from_agent = self.communicate_with_other_agents(
                        current_agent_id, other_agents, self.incorrect_count
                    )
                    if solution_from_agent is not None:
                        logger.info(f"Step {step}: Received solution from another agent.")
                        return solution_from_agent

            logger.warning("Maximum iterations reached without finding a confident solution.")
            return working_grid

        except Exception as e:
            logger.error(f"Error in solve_grid_task_with_multi_agent_support: {e}")
            return input_grid  # Return the original grid in case of failure

    def select_and_apply_action(self, step: int) -> Optional[int]:
        action = self.select_action(self.current_features)
        if action is None:
            logger.warning(f"Step {step}: No valid action selected.")
            return None
        logger.info(f"Step {step}: Selected action {action}.")
        working_grid = self.apply_action(self.current_features, action)
        return action

    def should_communicate_with_agents(self, step: int) -> bool:
        return step > 1 and self.incorrect_count > self.agent_communication_threshold

    def request_solution_from_agents(
        self,
        current_agent_id: int,
        other_agents: List['ProblemSolvingIntelligence']
    ) -> Optional[np.ndarray]:
        logger.info("Requesting solution from other agents.")
        return self.communicate_with_other_agents(current_agent_id, other_agents, self.incorrect_count)

    def communication_with_other_agents(self, current_agent_id: int, other_agents: List['ProblemSolvingIntelligence'], failures: int) -> Optional[np.ndarray]:
        try:
            if failures >= self.agent_communication_threshold:
                logger.info(f"Agent {current_agent_id} has failed {failures} times. Asking other agents for help...")
                for agent in other_agents:
                    if agent.has_solved_task():
                        logger.info(f"Agent {agent.get_id()} has solved the task. Copying solution.")
                        return agent.get_current_solution()
            return None
        except Exception as e:
            logging.error(f"Error in communication_with_other_agents: {e}")
            return None

    def get_current_solution(self) -> np.ndarray:
        try:
            return self.current_features  # Simplified
        except Exception as e:
            logging.error(f"Error in get_current_solution: {e}")
            return np.zeros(self.feature_length)

    # Multi-Agent ID for Logging
    def get_id(self) -> int:
        try:
            return id(self)
        except Exception as e:
            logging.error(f"Error in get_id: {e}")
            return -1

    def has_solved_task(self) -> bool:
        try:
            return self.last_task_solved
        except Exception as e:
            logging.error(f"Error in has_solved_task: {e}")
            return False

    @staticmethod
    def check_answer_correctness(predicted_action, correct_action, tolerance=0) -> bool:
        if isinstance(predicted_action, (int, float)) and isinstance(correct_action, (int, float)):
            return abs(predicted_action - correct_action) <= tolerance
        elif isinstance(predicted_action, np.ndarray) and isinstance(correct_action, np.ndarray):
            return np.allclose(predicted_action, correct_action, atol=tolerance)
        else:
            return predicted_action == correct_action

    def error_driven_action_selection(self, current_grid: np.ndarray, target_grid: np.ndarray) -> List[Tuple]:
        try:
            assert current_grid.shape == target_grid.shape, "current_grid and target_grid must have the same shape"
            mismatched_positions = np.argwhere(current_grid != target_grid)
            severity_map = {}

            for pos in mismatched_positions:
                x, y = pos
                current_value = current_grid[x, y]
                target_value = target_grid[x, y]
                severity_map[(x, y)] = abs(current_value - target_value)

            sorted_severity = sorted(severity_map.items(), key=lambda item: item[1], reverse=True)

            prioritized_actions = []
            for (x, y), _ in sorted_severity:
                target_color = target_grid[x, y]
                action = ('place_color', (int(x), int(y), int(target_color), 0))  # Assume layer 0 for simplicity
                prioritized_actions.append(action)

            return prioritized_actions
        except Exception as e:
            logging.error(f"Error in error_driven_action_selection: {e}")
            return []

    def evaluate_solution(self, working_grid: np.ndarray, correct_output: np.ndarray) -> bool:
        try:
            if np.array_equal(working_grid, correct_output):
                self.right_count += 1
                self.last_task_solved = True
                return True
            else:
                self.fail_count += 1
                self.last_task_solved = False
                return False
        except Exception as e:
            logging.error(f"Error in evaluate_solution: {e}")
            return False

    def update_context(self, current_grid: np.ndarray, target_grid: np.ndarray) -> None:
        if current_grid.shape != target_grid.shape:
            # Decide whether to resize or handle differently
            print(f"Grid shapes differ. Skipping context update for task.")
            self.current_context = None  # Or handle accordingly
            return

        mismatched_positions = np.argwhere(current_grid != target_grid)
        self.current_context = mismatched_positions
        logger.debug(f"Updated context with mismatched positions: {self.current_context}")

    def plot_distress_over_time(self, distress_history: np.ndarray) -> None:
        try:
            num_iterations, num_elements = distress_history.shape
            for i in range(num_elements):
                plt.plot(range(num_iterations), distress_history[:, i], label=f'Element {i}')

            plt.xlabel('Iteration')
            plt.ylabel('Distress Level')
            plt.title('Distress Level Over Time')
            plt.legend()
            plt.savefig('distress_plot.png')
            plt.close()
            logger.info("Distress plot saved as 'distress_plot.png'.")
        except Exception as e:
            logging.error(f"Error in plot_distress_over_time: {e}")

    def plot_training_metrics(self, rewards_history: List[float], q_values_history: List[float], epsilon_history: List[float]) -> None:
        try:
            plt.figure(figsize=(12, 4))

            # Plot Rewards
            plt.subplot(1, 3, 1)
            plt.plot(rewards_history, label='Rewards')
            plt.xlabel('Iteration')
            plt.ylabel('Reward')
            plt.title('Rewards Over Time')
            plt.legend()

            # Plot Q-values (average)
            plt.subplot(1, 3, 2)
            plt.plot(q_values_history, label='Average Q-value', color='orange')
            plt.xlabel('Iteration')
            plt.ylabel('Average Q-value')
            plt.title('Q-values Over Time')
            plt.legend()

            # Plot Epsilon
            plt.subplot(1, 3, 3)
            plt.plot(epsilon_history, label='Epsilon', color='green')
            plt.xlabel('Iteration')
            plt.ylabel('Epsilon Value')
            plt.title('Epsilon Decay Over Time')
            plt.legend()

            plt.tight_layout()
            plt.savefig('training_metrics.png')
            plt.close()
            logger.info("Training metrics plot saved as 'training_metrics.png'.")
        except Exception as e:
            logging.error(f"Error in plot_training_metrics: {e}")

    def train(self, training_data: Dict) -> None:
        logging.info("Starting training...")

        # Handle both dict and list data structures
        if isinstance(training_data, dict):
            iterable = training_data.items()
        elif isinstance(training_data, list):
            iterable = enumerate(training_data)
        else:
            logging.error("training_data is neither a list nor a dict")
            return

        for task_id, task in tqdm(iterable, desc="Training Tasks"):
            logging.info(f"Training on task: {task_id}")

            if not isinstance(task, dict):
                logging.error(f"Task {task_id} is not a dictionary. Skipping...")
                continue

            if 'train' not in task:
                logging.error(f"'train' key not found in task {task_id}. Skipping...")
                continue

            for idx, train_example in enumerate(tqdm(task['train'], desc=f"Training Task {task_id}", leave=False)):
                if not isinstance(train_example, dict):
                    logging.error(f"Train example {idx} in task {task_id} is not a dictionary. Skipping...")
                    continue

                if 'input' not in train_example or 'output' not in train_example:
                    logging.error(f"'input' or 'output' key not found in train example {idx} of task {task_id}. Skipping...")
                    continue

                input_grid = np.array(train_example['input'], dtype=int)
                output_grid = np.array(train_example['output'], dtype=int)

                # Validate correct_output
                if output_grid.size == 0:
                    logging.error(f"Output grid for task {task_id}, example {idx} is empty. Skipping...")
                    continue
                if not np.any(output_grid):
                    logging.error(f"Output grid for task {task_id}, example {idx} contains all zeros. Skipping...")
                    continue

                logger.debug(f"Output grid for task {task_id}, example {idx}:\n{output_grid}")

                # Ensure grids have the same shape
                if input_grid.shape != output_grid.shape:
                    logger.warning(f"Resizing input_grid from {input_grid.shape} to match output_grid {output_grid.shape}")
                    input_grid = self.resize_grid(input_grid, output_grid.shape)

                # Set current_grid and correct_output
                self.current_grid = input_grid.copy()
                self.correct_output = output_grid.copy()

                # Determine grid dimensions based on current_grid
                grid_height, grid_width = self.current_grid.shape[:2]
                num_layers = self.current_grid.shape[2] if self.current_grid.ndim == 3 else 1

                # Generate action space based on grid dimensions and current grid
                self.generate_action_space(grid_width, grid_height, num_layers)

                # Extract features
                input_features = self.extract_features(input_grid)
                output_features = self.extract_features(output_grid)

                # Encode the features in memory
                self.memory_system.memory_encoding(input_features)
                self.memory_system.memory_encoding(output_features)

                # Solve the grid task and generate the predicted output
                predicted_output = self.solve_grid_task_with_cognitive_refinement(input_grid, output_grid)

                # Check if predicted_output is valid
                if predicted_output is None:
                    logging.warning(f"Predicted output is None for task {task_id}. Skipping to next example.")
                    continue  # Skip to the next training example

                predicted_output = np.round(predicted_output).astype(int)

                # Handle shape mismatch between predicted and output grids
                if predicted_output.shape != output_grid.shape:
                    print(f"Shape mismatch for task {task_id}: resizing predicted output.")
                    predicted_output = self.resize_grid(predicted_output, output_grid.shape)

                # Update the context based on grid comparison (only if shapes match)
                if input_grid.shape == output_grid.shape:
                    self.update_context(input_grid, output_grid)
                else:
                    # Handle context update when shapes differ (if applicable)
                    pass

                # Select action based on the updated context
                action = self.select_action(input_features)  # Pass only state_features

                # Get the corresponding action ID
                action_id = self.INVERSE_ACTION_MAP.get(action)
                if action_id is None:
                    logging.warning(f"Action {action} not found in INVERSE_ACTION_MAP.")
                    continue  # Skip to the next iteration or handle appropriately

                # Calculate reward based on the accuracy of the predicted output
                if predicted_output.shape == output_grid.shape:
                    reward = np.sum(predicted_output == output_grid) / np.prod(output_grid.shape)
                else:
                    # Assign a lower reward or negative reward if shapes differ
                    reward = -1.0  # Penalty for incorrect grid shape

                # Perform Q-learning update based on the selected action and obtained reward
                self.RL.q_learning_update(input_features, action_id, reward, output_features)

                # Decay epsilon after each training example
                self.RL.decay_epsilon()
                self.epsilon_control.decay_epsilon()

    def evaluation(self, evaluation_data: Union[Dict, List], correct_outputs: Union[Dict, List]) -> None:
            """
            Evaluate the agent's performance on the provided evaluation data.

            :param evaluation_data: The data to evaluate, can be a dict or list.
            :param correct_outputs: The correct outputs corresponding to the evaluation data, can be a dict or list.
            """
            self.right_count = 0
            self.fail_count = 0

            logging.info("Starting evaluation...")
            self.is_evaluation = True  # Indicate that evaluation phase has started

            # Determine the type of evaluation_data and correct_outputs
            if isinstance(evaluation_data, dict) and isinstance(correct_outputs, dict):
                iterable = evaluation_data.items()
                correct_output_iterable = correct_outputs
            elif isinstance(evaluation_data, list) and isinstance(correct_outputs, list):
                iterable = enumerate(evaluation_data)
                correct_output_iterable = correct_outputs
            else:
                logging.error("Mismatch in types of evaluation_data and correct_outputs. Both should be dicts or both should be lists.")
                self.is_evaluation = False
                return

            for task_id, task in tqdm(iterable, desc="Evaluation Tasks"):
                logging.info(f"Evaluating task: {task_id}")

                if not isinstance(task, dict):
                    logging.error(f"Task {task_id} is not a dictionary. Skipping...")
                    self.fail_count += 1
                    continue

                if 'test' not in task:
                    logging.error(f"'test' key not found in task {task_id}. Skipping...")
                    self.fail_count += 1
                    continue

                for idx, eval_example in enumerate(tqdm(task['test'], desc=f"Evaluating Task {task_id}", leave=False)):
                    if not isinstance(eval_example, dict):
                        logging.error(f"Evaluation example {idx} in task {task_id} is not a dictionary. Skipping...")
                        self.fail_count += 1
                        continue

                    if 'input' not in eval_example:
                        logging.error(f"'input' key not found in evaluation example {idx} of task {task_id}. Skipping...")
                        self.fail_count += 1
                        continue

                    input_grid = np.array(eval_example['input'], dtype=int)

                    # Retrieve the correct output based on the structure of correct_outputs
                    try:
                        if isinstance(correct_output_iterable, dict):
                            # Access using task_id as key
                            if task_id not in correct_output_iterable:
                                logging.error(f"Task ID {task_id} not found in correct_outputs. Skipping...")
                                self.fail_count += 1
                                continue

                            task_correct_output = correct_output_iterable[task_id]

                            if isinstance(task_correct_output, dict):
                                # Access using string index
                                idx_key = str(idx)
                                if idx_key not in task_correct_output:
                                    logging.error(f"Index {idx_key} not found in correct_outputs for task {task_id}. Skipping...")
                                    self.fail_count += 1
                                    continue
                                correct_output = np.array(task_correct_output[idx_key], dtype=int)
                            elif isinstance(task_correct_output, list):
                                # Access using integer index
                                if idx >= len(task_correct_output):
                                    logging.error(f"Index {idx} out of range in correct_outputs for task {task_id}. Skipping...")
                                    self.fail_count += 1
                                    continue
                                correct_output = np.array(task_correct_output[idx], dtype=int)
                            else:
                                logging.error(f"Unsupported structure for correct_outputs[{task_id}]. Skipping...")
                                self.fail_count += 1
                                continue

                        elif isinstance(correct_output_iterable, list):
                            # Access using integer task_id
                            if task_id >= len(correct_output_iterable):
                                logging.error(f"Task ID {task_id} out of range in correct_outputs list. Skipping...")
                                self.fail_count += 1
                                continue

                            task_correct_output = correct_output_iterable[task_id]

                            if isinstance(task_correct_output, list):
                                if idx >= len(task_correct_output):
                                    logging.error(f"Index {idx} out of range in correct_outputs for task {task_id}. Skipping...")
                                    self.fail_count += 1
                                    continue
                                correct_output = np.array(task_correct_output[idx], dtype=int)
                            else:
                                logging.error(f"Unsupported structure for correct_outputs[{task_id}]. Skipping...")
                                self.fail_count += 1
                                continue
                        else:
                            logging.error("correct_outputs is neither a dict nor a list.")
                            self.fail_count += 1
                            continue

                    except Exception as e:
                        logging.error(f"Error accessing correct_output for task {task_id}, index {idx}: {e}")
                        self.fail_count += 1
                        continue

                    # Resize input_grid if necessary
                    if input_grid.shape != correct_output.shape:
                        logging.warning(f"Resizing input_grid from {input_grid.shape} to match correct_output {correct_output.shape}")
                        input_grid = self.resize_grid(input_grid, correct_output.shape)

                    # Solve the grid task and generate the predicted output
                    predicted_output = self.solve_grid_task_with_cognitive_refinement(input_grid, correct_output)
                    if predicted_output is None:
                        logging.warning(f"Predicted output is None for task {task_id}, example {idx}. Skipping...")
                        self.fail_count += 1
                        continue

                    predicted_output = np.round(predicted_output).astype(int)

                    # Handle shape mismatch between predicted and output grids
                    if predicted_output.shape != correct_output.shape:
                        logging.warning(f"Shape mismatch for task {task_id}, example {idx}: resizing predicted output.")
                        predicted_output = self.resize_grid(predicted_output, correct_output.shape)

                    # Evaluate the solution
                    is_correct = self.evaluate_solution(predicted_output, correct_output)

                    if is_correct:
                        logging.info(f"Task {task_id}, Example {idx}: Correct")
                        self.right_count += 1
                    else:
                        logging.info(f"Task {task_id}, Example {idx}: Incorrect")
                        self.fail_count += 1

                    # Log counts after each evaluation
                    logging.debug(f"Current counts - Right: {self.right_count}, Fail: {self.fail_count}")

            # Calculate and log final accuracy
            total_attempts = self.right_count + self.fail_count
            logging.info(f"Total correct tasks: {self.right_count}")
            logging.info(f"Total failed tasks: {self.fail_count}")

            if total_attempts > 0:
                accuracy = (self.right_count / total_attempts) * 100  # Convert to percentage
            else:
                accuracy = 0.0
            logging.info(f"Evaluation complete. Accuracy: {accuracy:.2f}%")
            print(f"Final Evaluation Accuracy: {accuracy:.2f}%")

            # Reset the evaluation flag
            self.is_evaluation = False

    # ===================
    # Main Execution
    # ===================

    def main():
        try:
            # Define the paths to the required JSON files
            training_challenges_path = os.path.join("arc-agi_training_challenges.json")
            evaluation_challenges_path = os.path.join("arc-agi_evaluation_challenges.json")
            evaluation_solutions_path = os.path.join("arc-agi_evaluation_solutions.json")

            # Check if files exist
            required_files = [
                (training_challenges_path, "Training challenges"),
                (evaluation_challenges_path, "Evaluation challenges"),
                (evaluation_solutions_path, "Evaluation solutions")
            ]
            all_files_exist = True
            for path, name in required_files:
                if not os.path.isfile(path):
                    logging.error(f"{name} file not found at {path}")
                    all_files_exist = False
            if not all_files_exist:
                logging.error("One or more required files are missing. Exiting main.")
                return

            # Load JSON files
            training_challenges = load_json_data(training_challenges_path)
            evaluation_challenges = load_json_data(evaluation_challenges_path)
            evaluation_solutions = load_json_data(evaluation_solutions_path)

            if training_challenges is None or evaluation_challenges is None or evaluation_solutions is None:
                logging.error("Failed to load all JSON data. Exiting.")
                return

            print(f"Loaded {len(training_challenges)} training challenges.")
            print(f"Loaded {len(evaluation_challenges)} evaluation challenges.")
            print(f"Loaded {len(evaluation_solutions)} evaluation solutions.")

            # Debugging: print the type of training_challenges
            print(f"type(training_challenges): {type(training_challenges)}")
            if isinstance(training_challenges, dict):
                print(f"Keys in training_challenges: {list(training_challenges.keys())[:5]}")
            elif isinstance(training_challenges, list):
                print(f"First training challenge: {training_challenges[0]}")

            feature_length = TARGET_FEATURE_LENGTH
            max_iterations = MAX_ITERATIONS

            # Initialize the ProblemSolvingIntelligence model
            psi = ProblemSolvingIntelligence(feature_length=feature_length, max_iterations=max_iterations)
            psi.train(training_challenges)
            psi.evaluation(evaluation_challenges, evaluation_solutions)
            psi.plot_training_metrics(psi.rewards_history, psi.q_values_history, psi.epsilon_history)
            psi.plot_distress_over_time(psi.distress_history)

            print("Evaluation complete.")

        except Exception as e:
            logging.error(f"Error in main: {e}")

    if __name__ == "__main__":
        main()


2024-10-27 10:03:27,613 - INFO - Imports completed successfully
2024-10-27 10:03:27,767 - INFO - Successfully loaded arc-agi_training_challenges.json


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
scipy version: 1.13.1
sklearn version: 1.5.2


2024-10-27 10:03:27,935 - INFO - Successfully loaded arc-agi_evaluation_challenges.json
2024-10-27 10:03:27,964 - INFO - Successfully loaded arc-agi_evaluation_solutions.json
2024-10-27 10:03:27,970 - INFO - Starting training...


Loaded 400 training challenges.
Loaded 400 evaluation challenges.
Loaded 400 evaluation solutions.
type(training_challenges): <class 'dict'>
Keys in training_challenges: ['007bbfb7', '00d62c1b', '017c7c7b', '025d127b', '045e512c']
ProblemSolvingIntelligence initialized


Training Tasks:   0%|          | 0/400 [00:00<?, ?it/s]2024-10-27 10:03:27,978 - INFO - Training on task: 007bbfb7

Training Task 007bbfb7:   0%|          | 0/5 [00:00<?, ?it/s]2024-10-27 10:03:27,986 - WARNING - Resizing input_grid from (3, 3) to match output_grid (9, 9)
2024-10-27 10:03:27,988 - INFO - Generating action space for grid (9x9), num_layers=1
2024-10-27 10:03:27,990 - INFO - Generated 56 actions.
2024-10-27 10:03:29,537 - INFO - Total patterns stored in Hopfield Network: 1
2024-10-27 10:03:30,092 - INFO - Total patterns stored in Hopfield Network: 2
2024-10-27 10:03:30,102 - INFO - Generating action space for grid (9x9), num_layers=1
2024-10-27 10:03:30,104 - INFO - Generated 56 actions.


Starting task...
Iteration 1
Selected action: ('place_color', (0, 2, 0, 0))
Iteration 2
Selected action: ('place_color', (2, 1, 0, 0))
Iteration 3
Selected action: ('place_color', (7, 4, 7, 0))
Iteration 4
Selected action: ('place_color', (7, 4, 7, 0))
Iteration 5
Selected action: ('place_color', (0, 8, 7, 0))
Iteration 6
Selected action: ('place_color', (0, 4, 7, 0))
Iteration 7
Selected action: ('place_color', (1, 7, 7, 0))
Iteration 8
Selected action: ('place_color', (5, 4, 7, 0))
Iteration 9
Selected action: ('place_color', (1, 6, 7, 0))
Iteration 10
Selected action: ('place_color', (6, 7, 7, 0))
Iteration 11
Selected action: ('place_color', (1, 8, 7, 0))
Iteration 12
Selected action: ('place_color', (2, 4, 7, 0))
Iteration 13
Selected action: ('place_color', (0, 1, 0, 0))
Iteration 14
Selected action: ('place_color', (1, 4, 7, 0))
Iteration 15
Selected action: ('place_color', (3, 1, 7, 0))
Iteration 16
Selected action: ('place_color', (1, 7, 7, 0))
Iteration 17
Selected action: ('

... (5,432 outputs left out)

2024-10-27 10:39:39,421 - INFO - Generated 192 actions.


Iteration 1
Selected action: ('place_color', (0, 7, 2, 0))
Iteration 2
Selected action: ('place_color', (5, 10, 2, 0))
Iteration 3
Selected action: ('place_color', (16, 14, 2, 0))
Iteration 4
Selected action: ('place_color', (7, 6, 2, 0))
Iteration 5
Selected action: ('place_color', (27, 8, 2, 0))
Iteration 6
Selected action: ('place_color', (2, 9, 2, 0))
Iteration 7
Selected action: ('place_color', (0, 5, 2, 0))
Iteration 8
Selected action: ('place_color', (14, 17, 2, 0))
Iteration 9
Selected action: ('place_color', (8, 7, 2, 0))
Iteration 10
Selected action: ('place_color', (7, 7, 2, 0))
Iteration 11
Selected action: ('place_color', (24, 3, 2, 0))
Iteration 12
Selected action: ('place_color', (2, 8, 2, 0))
Iteration 13
Selected action: ('place_color', (24, 5, 2, 0))
Iteration 14
Selected action: ('place_color', (1, 1, 2, 0))
Iteration 15
Selected action: ('place_color', (17, 13, 2, 0))
Iteration 16
Selected action: ('place_color', (22, 3, 2, 0))
Iteration 17
Selected action: ('place_

2024-10-27 10:39:41,849 - INFO - Task ff72ca3e, Example 0: Incorrect

Evaluation Tasks: 100%|██████████| 400/400 [07:01<00:00,  1.05s/it]
2024-10-27 10:39:41,865 - INFO - Total correct tasks: 840
2024-10-27 10:39:41,867 - INFO - Total failed tasks: 417
2024-10-27 10:39:41,869 - INFO - Evaluation complete. Accuracy: 66.83%
2024-10-27 10:39:41,870 - ERROR - Error in main: 'ProblemSolvingIntelligence' object has no attribute 'rewards_history'


Iteration 198
Selected action: ('place_color', (0, 2, 2, 0))
Iteration 199
Selected action: ('place_color', (17, 16, 2, 0))
Iteration 200
Selected action: ('place_color', (23, 6, 2, 0))
Reached the maximum total iterations (200).
Final Evaluation Accuracy: 66.83%
